# RedisVectorStore

This notebook demonstrates the usage of RedisVectorStore from the langchain-redis package. RedisVectorStore leverages Redis as a vector database, enabling efficient storage, retrieval, and similarity search of vector embeddings.

## Installation

First, we need to install the necessary packages. Run the following command to install langchain-redis, sentence-transformers, and scikit-learn:

In [ ]:
!pip install -U langchain-redis sentence-transformers scikit-learn

## Importing Required Libraries
We'll import the necessary libraries for our tasks:

In [ ]:
import os
from langchain_redis import RedisVectorStore
from sentence_transformers import SentenceTransformer
from sklearn.datasets import fetch_20newsgroups
from langchain.docstore.document import Document
from langchain_core.embeddings import Embeddings

## Setting up Redis Connection
To use RedisVectorStore, you need a running Redis instance. For this example, we assume a local Redis instance running on the default port. Modify the URL if your setup differs:

In [ ]:
import os

# Use the environment variable if set, otherwise default to localhost
REDIS_URL = os.getenv("REDIS_URL", "redis://localhost:6379")
print(f"Connecting to Redis at: {REDIS_URL}")

## Preparing Sample Data
We'll use a subset of the 20 Newsgroups dataset for this demonstration. This dataset contains newsgroup posts on various topics. We'll focus on two categories: 'alt.atheism' and 'sci.space':

In [ ]:
categories = ['alt.atheism', 'sci.space']
newsgroups = fetch_20newsgroups(subset='train', categories=categories, shuffle=True, random_state=42)

# Use only the first 100 documents to keep the example manageable
texts = newsgroups.data[:100]
metadata = [{"category": newsgroups.target_names[target]} for target in newsgroups.target[:100]]

documents = [Document(page_content=text, metadata=meta) for text, meta in zip(texts, metadata)]

## Creating Embeddings
We'll use the SentenceTransformer model to create embeddings. This model runs locally and doesn't require an API key:

In [ ]:
from typing import List

class SentenceTransformerEmbeddings(Embeddings):
    def __init__(self, model_name="all-MiniLM-L6-v2"):
        self.model = SentenceTransformer(model_name)

    def embed_documents(self, texts: List[str]) -> List[List[float]]:
        embeddings = self.model.encode(texts)
        return embeddings.tolist()

    def embed_query(self, text: str) -> List[float]:
        embedding = self.model.encode(text)
        return embedding.tolist()

embeddings = SentenceTransformerEmbeddings()

## Basic Usage with LangChain's RedisVectorStore
Now we'll demonstrate basic usage of RedisVectorStore, including creating an instance, inserting data, and performing a simple similarity search.

### Creating a RedisVectorStore instance and inserting data
We'll create a RedisVectorStore instance and populate it with our sample data:

In [ ]:
vector_store = RedisVectorStore.from_documents(
    documents,
    embeddings,
    redis_url=REDIS_URL,
    index_name="newsgroups",
)

### Performing a simple similarity search
Let's perform a basic similarity search using a query about space exploration:

In [ ]:
query = "Tell me about space exploration"
results = vector_store.similarity_search(query, k=2)

print("Simple Similarity Search Results:")
for doc in results:
    print(f"Content: {doc.page_content[:100]}...")
    print(f"Metadata: {doc.metadata}")
    print()

## Advanced Queries with RedisVectorStore
RedisVectorStore supports more advanced query types. We'll demonstrate similarity search with metadata filtering, maximum marginal relevance search, and similarity search with score.

### Similarity search with metadata filtering
We can filter our search results based on metadata:

In [ ]:
from redisvl.query.filter import Tag

query = "Tell me about space exploration"

# Create a filter expression
filter_condition = Tag("category") == "sci.space"

filtered_results = vector_store.similarity_search(query, k=2, filter=filter_condition)

print("Filtered Similarity Search Results:")
for doc in filtered_results:
    print(f"Content: {doc.page_content[:100]}...")
    print(f"Metadata: {doc.metadata}")
    print()

### Maximum marginal relevance search
Maximum marginal relevance search helps in getting diverse results:

In [ ]:
# Maximum marginal relevance search with filter
mmr_results = vector_store.max_marginal_relevance_search(
    query, k=2, fetch_k=10, filter=filter_condition
)

print("Maximum Marginal Relevance Search Results:")
for doc in mmr_results:
    print(f"Content: {doc.page_content[:100]}...")
    print(f"Metadata: {doc.metadata}")
    print()

### Similarity search with score
We can also get similarity scores along with our search results:

In [ ]:
# Similarity search with score and filter
scored_results = vector_store.similarity_search_with_score(
    query, k=2, filter=filter_condition
)

print("Similarity Search with Score Results:")
for doc, score in scored_results:
    print(f"Content: {doc.page_content[:100]}...")
    print(f"Metadata: {doc.metadata}")
    print(f"Score: {score}")
    print()

## Introduction to RedisVL
The Redis LangChain integration is powered by the Redis Vector Library (RedisVL). RedisVL provides advanced capabilities for working with vector data in Redis, including custom schemas and hybrid queries.

## Advanced Usage with RedisVL
We'll now demonstrate more advanced usage with RedisVL, including creating a custom schema and performing hybrid queries.

In [ ]:
from redisvl.index import SearchIndex
from redisvl.query import VectorQuery
from redisvl.query.filter import Text, Tag

### Defining a custom schema
We'll define a custom schema for our newsgroup data:

In [ ]:
schema = {
    "index": {
        "name": "newsgroups_custom",
        "prefix": "newsgroup:",
        "storage_type": "hash",
    },
    "fields": [
        {"name": "text", "type": "text"},
        {"name": "category", "type": "tag"},
        {
            "name": "embedding",
            "type": "vector",
            "attrs": {
                "dims": 384,  # Dimension of the SentenceTransformer embeddings
                "distance_metric": "cosine",
                "algorithm": "flat"
            }
        }
    ]
}

### Creating an index with RedisVL
Now we'll create an index using our custom schema:

In [ ]:
index = SearchIndex.from_dict(schema)
index.connect(REDIS_URL)
index.create(overwrite=True)

### Inserting data using RedisVL
We'll insert our data into the new index:

In [ ]:
data = [
    {
        "text": doc.page_content,
        "category": doc.metadata["category"],
        "embedding": embeddings.embed_query(doc.page_content)
    }
    for doc in documents
]

index.load(data)

### Performing a hybrid query
Finally, we'll perform a hybrid query that combines vector similarity search with metadata filtering:

In [ ]:
query_embedding = embeddings.embed_query(query)

hybrid_query = VectorQuery(
    vector=query_embedding,
    vector_field_name="embedding",
    return_fields=["text", "category"],
    num_results=2,
    filter_expression=Tag("category") == "sci.space"
)

results = index.query(hybrid_query)

print("Hybrid Query Results:")
for result in results:
    print(f"Content: {result['text'][:100]}...")
    print(f"Category: {result['category']}")
    print(f"Score: {result['vector_distance']}")
    print()

## Cleanup
After we're done, it's important to clean up our Redis indices:

In [ ]:
# Delete the LangChain index
vector_store.delete_keys()

# Delete the RedisVL index
index.delete()